In [7]:
from collections import Counter
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
tree = ET.parse('Block_1.xml')
root = tree.getroot()

In [8]:
''' O usuario fornecera os dados da seguinte forma[
[(Nome_elemento,tipo_elemento,subtipo_elemento,0),aresta_anterior, aresta_posterior]
[(Contato1,contato,1,0),0,1]
A aresta anterior deverá ser zero SEMPRE que o elemento estiver diretamente ligado ao barramento energizado
]'''
def diferenciar_nomes(dados_ladder):
    '''Funçao de pre processamento para retirar nomes repetidos e ordenar o problema'''
    contador = {}
    for x in dados_ladder:
        elemento = x[0]
        if elemento in contador:
            contador[elemento] +=1
        else:
            contador[elemento] = 1
        repeticoes = contador[elemento] -1
        x[0] = x[0][:-1] + (1*repeticoes,)
    return(dados_ladder)

In [2]:
dados_ladder_otimizados = [[
    [("cont1",'contato',"1",0), 0, 1],
    [("cont3",'contato',"1",0), 0, 2],
    [("cont4",'contato',"1",0), 0, 2],
    [("cont9",'contato',"1",0), 0, 4],
    [("cont9",'contato',"1",0), 1, 2],
    [("cont10",'contato',"1",0), 4, 3],
    [("cont6",'contato',"1",0), 3, 5],
    [("bobina1",'contato',"1",0), 5, 6],
    [("cont7",'contato',"1",0), 3, 7],
    [("cont8",'contato',"1",0), 7, 8],
    [("bobina2",'contato',"1",0), 8, 9],
    [("cont5",'contato',"1",0), 2, 3]  # <--- O elemento central jogado no final da lista
]]


In [4]:
dados_ladder_otimizados = diferenciar_nomes(dados_ladder_otimizados)
dados_ladder_otimizados



[[('cont1', 'contato', '1', 0), 0, 1],
 [('cont3', 'contato', '1', 0), 0, 2],
 [('cont4', 'contato', '1', 0), 0, 2],
 [('cont9', 'contato', '1', 0), 0, 4],
 [('cont9', 'contato', '1', 1), 1, 2],
 [('cont10', 'contato', '1', 0), 4, 3],
 [('cont6', 'contato', '1', 0), 3, 5],
 [('bobina1', 'contato', '1', 0), 5, 6],
 [('cont7', 'contato', '1', 0), 3, 7],
 [('cont8', 'contato', '1', 0), 7, 8],
 [('bobina2', 'contato', '1', 0), 8, 9],
 [('cont5', 'contato', '1', 0), 2, 3]]

In [5]:
# ==========================================
# ÁREA DE TESTE
# ==========================================
# Formato Otimizado: [Elemento, Nó_Anterior, Nó_Posterior]
# Vou manter a lista bagunçada para provar que a matemática organiza sozinha!
dados_ladder_otimizados = [
    ["cont1", 0, 1],
    ["cont3", 0, 2],
    ["cont4", 0, 2],
    ["cont9_inf", 0, 4],
    ["cont9_sup", 1, 2],
    ["cont10", 4, 3],
    ["cont6", 3, 5],
    ["bobina1", 5, 6],
    ["cont7", 3, 7],
    ["cont8", 7, 8],
    ["bobina2", 8, 9],
    ["cont5", 2, 3]  # <--- O elemento central jogado no final da lista
]

# Executando a função
elementos_nomeados = gerar_chaves_ladder(dados_ladder_otimizados)

'''# Imprimindo o resultado formatado
print(f"{'CHAVE':<7} | {'ELEMENTO':<12} | {'LINHA VIRTUAL':<15} | {'CONEXÃO (In -> Out)'}")
print("-" * 65)
for item in elementos_nomeados:
    print(f" {item['Chave']:02d}     | {item['Elemento']:<12} | Linha {item['Linha_Virtual']:<10} | Nó {item['No_in']} -> Nó {item['No_out']}")'''

'# Imprimindo o resultado formatado\nprint(f"{\'CHAVE\':<7} | {\'ELEMENTO\':<12} | {\'LINHA VIRTUAL\':<15} | {\'CONEXÃO (In -> Out)\'}")\nprint("-" * 65)\nfor item in elementos_nomeados:\n    print(f" {item[\'Chave\']:02d}     | {item[\'Elemento\']:<12} | Linha {item[\'Linha_Virtual\']:<10} | Nó {item[\'No_in\']} -> Nó {item[\'No_out\']}")'

In [9]:
def gerar_sequencia_ladder_com_nos(lista_elementos):
    elementos_dict = {}
    
    # 1. REGISTRO [Nome, No_in, No_out]
    for i, el in enumerate(lista_elementos):
        elementos_dict[i] = {
            "nome": el[0], 
            "no_in": el[1], 
            "no_out": el[2],
            "predecessores": set(), 
            "sucessores": set(),
            "indice_original": i,
            "y_score": None 
        }

    # 2. MAPEAMENTO DOS FIOS
    saidas_por_no = {}
    for i, el in elementos_dict.items():
        if el["no_out"] not in saidas_por_no: 
            saidas_por_no[el["no_out"]] = []
        saidas_por_no[el["no_out"]].append(i)

    for i, el in elementos_dict.items():
        if el["no_in"] in saidas_por_no:
            for pred_id in saidas_por_no[el["no_in"]]:
                el["predecessores"].add(pred_id)
                elementos_dict[pred_id]["sucessores"].add(i)

    # 3. CÁLCULO DA LINHA VIRTUAL
    raizes = [i for i, el in elementos_dict.items() if len(el["predecessores"]) == 0]
    raizes.sort(key=lambda x: elementos_dict[x]["indice_original"])
    
    for linha, id_raiz in enumerate(raizes):
        elementos_dict[id_raiz]["y_score"] = linha

    grau_entrada_y = {i: len(el["predecessores"]) for i, el in elementos_dict.items()}
    fila_y = raizes[:]
    
    while fila_y:
        atual_id = fila_y.pop(0)
        atual = elementos_dict[atual_id]
        
        for suc_id in atual["sucessores"]:
            suc = elementos_dict[suc_id]
            if suc["y_score"] is None:
                suc["y_score"] = atual["y_score"]
            else:
                suc["y_score"] = min(suc["y_score"], atual["y_score"])
            
            grau_entrada_y[suc_id] -= 1
            if grau_entrada_y[suc_id] == 0:
                fila_y.append(suc_id)

    # 4. ORDENAÇÃO TOPOLÓGICA
    grau_entrada = {i: len(el["predecessores"]) for i, el in elementos_dict.items()}
    prontos = [i for i, el in elementos_dict.items() if grau_entrada[i] == 0]
    
    resultado = [] 
    
    while prontos:
        prontos.sort(key=lambda x: (elementos_dict[x]["y_score"], elementos_dict[x]["indice_original"]))
        
        atual_id = prontos.pop(0)
        atual = elementos_dict[atual_id]
        
        # AQUI ESTÁ A MUDANÇA: Guarda a sub-lista completa com nome e nós
        resultado.append([atual["nome"], atual["no_in"], atual["no_out"]])
        
        for suc_id in atual["sucessores"]:
            grau_entrada[suc_id] -= 1
            if grau_entrada[suc_id] == 0:
                prontos.append(suc_id)
                
    return resultado

# ==========================================
# TESTE DA NOVA SAÍDA
# ==========================================
'''dados_ladder_otimizados = [
    ["cont1", 0, 1],
    ["cont3", 0, 2],
    ["cont4", 0, 2],
    ["cont9_inf", 0, 4],
    ["cont9_sup", 1, 2],
    ["cont10", 4, 3],
    ["cont6", 3, 5],
    ["bobina1", 5, 6],
    ["cont7", 3, 7],
    ["cont8", 7, 8],
    ["bobina2", 8, 9],
    ["cont5", 2, 3]  
]

lista_ordenada = gerar_sequencia_ladder_com_nos(dados_ladder_otimizados)

# Imprimindo item por item para visualizar melhor a estrutura
print("Sua lista final para o gerador de XML:")
print("[\n  " + ",\n  ".join(str(item) for item in lista_ordenada) + "\n]")'''

'dados_ladder_otimizados = [\n    ["cont1", 0, 1],\n    ["cont3", 0, 2],\n    ["cont4", 0, 2],\n    ["cont9_inf", 0, 4],\n    ["cont9_sup", 1, 2],\n    ["cont10", 4, 3],\n    ["cont6", 3, 5],\n    ["bobina1", 5, 6],\n    ["cont7", 3, 7],\n    ["cont8", 7, 8],\n    ["bobina2", 8, 9],\n    ["cont5", 2, 3]  \n]\n\nlista_ordenada = gerar_sequencia_ladder_com_nos(dados_ladder_otimizados)\n\n# Imprimindo item por item para visualizar melhor a estrutura\nprint("Sua lista final para o gerador de XML:")\nprint("[\n  " + ",\n  ".join(str(item) for item in lista_ordenada) + "\n]")'

In [6]:
resultado = gerar_sequencia_ladder_com_nos(dados_ladder_otimizados)
resultado

[[('cont1', 'contato', '1', 0), 0, 1],
 [('cont9', 'contato', '1', 1), 1, 2],
 [('cont3', 'contato', '1', 0), 0, 2],
 [('cont4', 'contato', '1', 0), 0, 2],
 [('cont5', 'contato', '1', 0), 2, 3],
 [('cont9', 'contato', '1', 0), 0, 4],
 [('cont10', 'contato', '1', 0), 4, 3],
 [('cont6', 'contato', '1', 0), 3, 5],
 [('bobina1', 'contato', '1', 0), 5, 6],
 [('cont7', 'contato', '1', 0), 3, 7],
 [('cont8', 'contato', '1', 0), 7, 8],
 [('bobina2', 'contato', '1', 0), 8, 9]]

In [10]:
def ordenar_tipo_elemento(dados):
    #Com a lista ja ordenada, agora vou ordenar o elemento para colocalo na part do meu xml
    contagem = Counter([x[2] for x in dados])
    vistos = {}
    resultado = []

    for item in dados:
        valor = item[2]

        # conta quantas vezes já vi esse valor
        vistos[valor] = vistos.get(valor, 0) + 1

        resultado.append(item)

        # se for a ÚLTIMA ocorrência e tiver repetição
        if contagem[valor] > 1 and vistos[valor] == contagem[valor]:
            resultado.append(['card',valor, contagem[valor]])

    return resultado

def definir_uid_part(lista_ordenada_com_card,dados):
    '''
    O objetivo dessa funcao é adicionar uma lista com um ou dois elementos que serao os os UIds do nome do elemento e do tipo do elemento.
    Se for uma junção de cardinalidade, ele só recebera um elemento para ser colocado ao part   
    Retorna a lista ordenada com:
    |[(nome,tipo,subtipo,comtagem elememtos iguais),no_entrada,no_saida,[uid_nome,uid_contado]| se nao for card
    |[card,numero_elemento_repetido, quantidade_de_repeticoes,[uid_contato]]
    '''
    tamanho_lista =  len(lista_ordenada_com_card)
    tamanho_card = len(dados)
    uid_inicial = 21
    uid_part =  21 + tamanho_card
    for x in range(len(lista_ordenada_com_card)):
        if lista_ordenada_com_card[x][0] == 'card':
            lista_ordenada_com_card[x].append([])
            lista_ordenada_com_card[x][-1].append(uid_part)
            uid_part +=1  
        else:
            lista_ordenada_com_card[x].append([])
            lista_ordenada_com_card[x][-1].append(uid_inicial)
            lista_ordenada_com_card[x][-1].append(uid_part)
            uid_inicial +=1
            uid_part +=1
    uid_inicial = 21
    '''for x in range(len(dados)):
        dados[x].append([])
        dados[x][-1].append(uid_inicial)
        uid_inicial +=1'''

    return(lista_ordenada_com_card,dados,uid_part)



In [8]:
lo_com_card = ordenar_tipo_elemento(resultado)

In [9]:
pre_xml = definir_uid_part(lo_com_card,resultado)
pre_xml

([[('cont1', 'contato', '1', 0), 0, 1, [21, 33]],
  [('cont9', 'contato', '1', 1), 1, 2, [22, 34]],
  [('cont3', 'contato', '1', 0), 0, 2, [23, 35]],
  [('cont4', 'contato', '1', 0), 0, 2, [24, 36]],
  ['card', 2, 3, [37]],
  [('cont5', 'contato', '1', 0), 2, 3, [25, 38]],
  [('cont9', 'contato', '1', 0), 0, 4, [26, 39]],
  [('cont10', 'contato', '1', 0), 4, 3, [27, 40]],
  ['card', 3, 2, [41]],
  [('cont6', 'contato', '1', 0), 3, 5, [28, 42]],
  [('bobina1', 'contato', '1', 0), 5, 6, [29, 43]],
  [('cont7', 'contato', '1', 0), 3, 7, [30, 44]],
  [('cont8', 'contato', '1', 0), 7, 8, [31, 45]],
  [('bobina2', 'contato', '1', 0), 8, 9, [32, 46]]],
 [[('cont1', 'contato', '1', 0), 0, 1, [21, 33]],
  [('cont9', 'contato', '1', 1), 1, 2, [22, 34]],
  [('cont3', 'contato', '1', 0), 0, 2, [23, 35]],
  [('cont4', 'contato', '1', 0), 0, 2, [24, 36]],
  [('cont5', 'contato', '1', 0), 2, 3, [25, 38]],
  [('cont9', 'contato', '1', 0), 0, 4, [26, 39]],
  [('cont10', 'contato', '1', 0), 4, 3, [27, 4

In [11]:
lista_nomear = []
lista_de = []#de = definir elemento
dicionario_elementos = {"bobina":"coil","contato":"Contact"}
dicionario_bobinas = {"1":"coil","c":"coil","r":"RCoil","s":"SCoil"}
dicionario_contatos = {"1":"Contact","c":"contact"}
dicionario_elementos_completo = {"Contact":dicionario_contatos,"coil":dicionario_bobinas}

def parts_total_inicio():
    root = ET.Element("Parts")
    #xml_string = ET.tostring(root, encoding="utf-8")
    #lista_nomear.append(xml_string)



def nomear_elemento(uid,nome,tipo='1',subtipo='1',extra = 1):
    '''Cria o codigo xml para o nome do elemento'''
    access = ET.Element("Access", Scope ="GlobalVariable", UId = str(uid) )
    symbol = ET.SubElement(access, "Symbol")
    ET.SubElement(symbol, "Component", Name=str(nome))
    ET.indent(access, space="    ", level=0)
    xml_string = ET.tostring(access, encoding='unicode') +'\n'
    #print(xml_string)
    lista_nomear.append(xml_string)
    return('Resolvido')

def retornar_string_elemento(tipo,subtipo = '1'):
    string_elemento = dicionario_elementos_completo[dicionario_elementos[tipo]][subtipo]
    return(string_elemento)

def definir_elemento(uid,nome, tipo,subtipo = '1'):
    '''Cria o codigo xml para o tipo de elemento associado ao nome'''
    part = ET.Element("Part", Name = retornar_string_elemento(tipo,subtipo), UId = str(uid))
    xml_string_2 = ET.tostring(part, encoding='unicode') +'\n'
    #lista definir
    lista_de.append(xml_string_2)
    #print(xml_string_2)

def parte_esquerda(uid,numero_cardinalidade):
    '''
    Cria mais um #part referenciando ao objeto O de adição de elementos em um mesmo no
    Responsavel pela cardinalidade ao definir parts
    ATENCAO: PRECISO DEFINIR A  ENTRADA NUMERO_CARDINALIDADE QUE VEM DE UMA OUTRA LISTA QUE EU JA FIZ
    '''
    part = ET.Element("Part", Name="O", UId=str(uid))
    template_value = ET.SubElement(part, "TemplateValue", Name="Card", Type="Cardinality")
    template_value.text = str(numero_cardinalidade)
    ET.indent(part, space="    ", level=0)
    xml_string = ET.tostring(part, encoding='unicode') +'\n'
    lista_de.append(xml_string)
    #print(xml_string)
    #return('Fim')


def escrever_elementos_xml_part(lista_com_cardinalidade):
    parts_total_inicio()
    for x in range(len(lista_com_cardinalidade)):
        if lista_com_cardinalidade[x][0] != 'card':
            nomear_elemento(lista_com_cardinalidade[x][-1][0],lista_com_cardinalidade[x][0][0])
    for x in range(len(lista_com_cardinalidade)):
        if lista_com_cardinalidade[x][0] == 'card':
            parte_esquerda(lista_com_cardinalidade[x][-1][0],lista_com_cardinalidade[x][2])
        else:
            definir_elemento(lista_com_cardinalidade[x][-1][-1],lista_com_cardinalidade[x][0][0],lista_com_cardinalidade[x][0][1],lista_com_cardinalidade[x][-0][2])
    return(lista_de)

def parts_final(lista_sem_cardinalidade,lista_parts):
    lista_nomear.clear()
    lista_de.clear()
    nomes_diferentes = diferenciar_nomes(lista_sem_cardinalidade)
    resultado = gerar_sequencia_ladder_com_nos(nomes_diferentes)
    lista_ordenada_com_cardinalidade = ordenar_tipo_elemento(resultado)
    pre_xml = definir_uid_part(lista_ordenada_com_cardinalidade,resultado)
    #print(lista_ordenada_com_cardinalidade)
    lista_parts.append([])
    inicio_parts = ["<Parts>\n"]
    fim_parts = ["</Parts>\n"]
    meio_parts =lista_nomear + escrever_elementos_xml_part(pre_xml[0])
    lista_parts[-1] = inicio_parts + meio_parts + fim_parts
    #print(lista_parts[-1])
    print('--------DEBUG--------')
    return(pre_xml[0],lista_parts,pre_xml[2])
     

In [90]:
a = parts_final(dados_ladder_otimizados,[])
a[0]

--------DEBUG--------


[[('cont1', 'contato', '1', 0), 0, 1, [21, 33]],
 [('cont9', 'contato', '1', 1), 1, 2, [22, 34]],
 [('cont3', 'contato', '1', 0), 0, 2, [23, 35]],
 [('cont4', 'contato', '1', 0), 0, 2, [24, 36]],
 ['card', 2, 3, [37]],
 [('cont5', 'contato', '1', 0), 2, 3, [25, 38]],
 [('cont9', 'contato', '1', 0), 0, 4, [26, 39]],
 [('cont10', 'contato', '1', 0), 4, 3, [27, 40]],
 ['card', 3, 2, [41]],
 [('cont6', 'contato', '1', 0), 3, 5, [28, 42]],
 [('bobina1', 'contato', '1', 0), 5, 6, [29, 43]],
 [('cont7', 'contato', '1', 0), 3, 7, [30, 44]],
 [('cont8', 'contato', '1', 0), 7, 8, [31, 45]],
 [('bobina2', 'contato', '1', 0), 8, 9, [32, 46]]]

[['<Parts>', '<Access Scope="GlobalVariable" UId="21">\n    <Symbol>\n        <Component Name="cont1" />\n    </Symbol>\n</Access>', '<Access Scope="GlobalVariable" UId="22">\n    <Symbol>\n        <Component Name="cont9" />\n    </Symbol>\n</Access>', '<Access Scope="GlobalVariable" UId="23">\n    <Symbol>\n        <Component Name="cont3" />\n    </Symbol>\n</Access>', '<Access Scope="GlobalVariable" UId="24">\n    <Symbol>\n        <Component Name="cont4" />\n    </Symbol>\n</Access>', '<Access Scope="GlobalVariable" UId="25">\n    <Symbol>\n        <Component Name="cont5" />\n    </Symbol>\n</Access>', '<Access Scope="GlobalVariable" UId="26">\n    <Symbol>\n        <Component Name="cont9" />\n    </Symbol>\n</Access>', '<Access Scope="GlobalVariable" UId="27">\n    <Symbol>\n        <Component Name="cont10" />\n    </Symbol>\n</Access>', '<Access Scope="GlobalVariable" UId="28">\n    <Symbol>\n        <Component Name="cont6" />\n    </Symbol>\n</Access>', '<Access Scope="GlobalVari

In [83]:
contas = {
    "wendel" : {'senha':123, 'uid':123},
    "wesley" :{'senha':456,'uid':456}
}
contas["wendel"]['senha']

123

In [11]:
pre_xml[2]

47

In [ ]:
def juntar_xml_parts():
    """Junta as strings das listas e abraça tudo com a tag <Parts>"""
    
    # 1. Abre a tag no início do blocão
    xml_completo = "<Parts>\n"
    
    # 2. Adiciona todos os blocos <Access> gerados
    for trecho in lista_nomear:
        xml_completo += trecho + "\n"
        
    # 3. Adiciona todos os blocos <Part> gerados
    for trecho in lista_de:
        xml_completo += trecho + "\n"
        
    # 4.  Fecha a tag no final do blocão
    xml_completo += "</Parts>"
    
    return xml_completo

In [14]:
print(juntar_xml_parts())

<Parts>
<Access Scope="GlobalVariable" UId="21">
    <Symbol>
        <Component Name="cont1" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="22">
    <Symbol>
        <Component Name="cont9" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="23">
    <Symbol>
        <Component Name="cont3" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="24">
    <Symbol>
        <Component Name="cont4" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="25">
    <Symbol>
        <Component Name="cont5" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="26">
    <Symbol>
        <Component Name="cont9" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="27">
    <Symbol>
        <Component Name="cont10" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="28">
    <Symbol>
        <Component Name="cont6" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="29">
    <Symbol>
        <Component Name="bobina1"

In [13]:
escrever_elementos_xml_part(pre_xml[0])

<Access Scope="GlobalVariable" UId="21">
    <Symbol>
        <Component Name="cont1" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="22">
    <Symbol>
        <Component Name="cont9" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="23">
    <Symbol>
        <Component Name="cont3" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="24">
    <Symbol>
        <Component Name="cont4" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="25">
    <Symbol>
        <Component Name="cont5" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="26">
    <Symbol>
        <Component Name="cont9" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="27">
    <Symbol>
        <Component Name="cont10" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="28">
    <Symbol>
        <Component Name="cont6" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="29">
    <Symbol>
        <Component Name="bobina1" />
    

In [17]:
pre_xml[-1]

47

In [8]:
x = [1,2,3]
len(x)

3

In [12]:
'''
Bloco de implementação dos Wires
Necessário passar pelo bloco de implementação Part usando o UId que continuou do bloco anterior
'''
lista_wires = []
lista_confirmar_identificado = []
lista_conferencia =[]
lista_geral_de_fios = []
def barra_energizada(uid,lista_com_cardinalidade):
    #Funcao ligada ao powerrail. No futuro vou deixala como opcional para ser ativada ou não
    #lista_wires.append([])
    wire = ET.Element("Wire", UId = str(uid))
    ET.SubElement(wire,"Powerrail")
    #uid = uid + 1
    for x in range(len(lista_com_cardinalidade)):
        if lista_com_cardinalidade[x][0] != 'card' and lista_com_cardinalidade[x][1] == 0:
            ET.SubElement(wire,"NameCon", UId= str(lista_com_cardinalidade[x][-1][-1]) , Name="in")
    ET.indent(wire, space="  ")
    xml_output_w1 = ET.tostring(wire, encoding="unicode") +'\n'
    lista_wires.append(xml_output_w1)
    print(lista_wires)
    #return(lista_wires,uid)
    return(uid)

def referenciar_nome_tipo(uid,lista_elemento,lista_conferencia):
    #faz a relação entre o endereço do nome que eu coloquei e o o endereço do tipo de elemento que eu coloquei
    if lista_elemento[-1][-2] not in lista_conferencia:
        uid = uid + 1
        #Mesmo passo para os outros    
        wire = ET.Element("Wire", UId = str(uid))
        ET.SubElement(wire, "IdentCon", UId=str(lista_elemento[-1][-2]))
        ET.SubElement(wire, "NameCon", UId=str(lista_elemento[-1][-1]), Name="operand")
        ET.indent(wire, space="  ")
        xml_string = ET.tostring(wire, encoding="unicode")
        lista_wires.append(xml_string)
        lista_conferencia.append(lista_elemento[-1][-2])
    return(uid)

def fio_conect(uid,id_antes,id_depois): 
    '''
    Conecta dois elementos pelo seu código  de tipo.Só faz a conexão de dois elementos simples
    | |----------( )
    '''   
    uid += 1
    wire = ET.Element("Wire", UId = str(uid) )
    ET.SubElement(wire, "NameCon", UId= str(id_antes),Name="out")
    ET.SubElement(wire, "NameCon", UId=str(id_depois), Name="in")
    ET.indent(wire, space="  ")
    xml_string = ET.tostring(wire, encoding="unicode")
    lista_wires.append(xml_string)
    
    return(uid)


def fio_conect_varios_esquerda(uid,id_antes,id_depois,contador_fio_master):#Conectar varios elementos a um no
    '''
    Varias entradas, uma saida. A diferença para o anterior é o in, que vira in1,in2,...
    Nessa função, ao chamar o loop que vai aciona-la o que vai mudar é o in. O out é o no de cardinalidade que vira um elemento nao declarado
    '''
    uid +=1
    wire = ET.Element("Wire", UId = str(uid) )
    ET.SubElement(wire, "NameCon", UId=str(id_antes),Name="out")
    ET.SubElement(wire, "NameCon", UId=str(id_depois), Name="in"+str(contador_fio_master+1))
    ET.indent(wire, space="  ")
    ET.indent(wire, space="  ")
    xml_string = ET.tostring(wire, encoding="unicode")
    lista_wires.append(xml_string)
    return(uid)

def fio_conect_varios_direita(uid,id_antes,lista_depois):
    '''
    Exatamente a mesma coisa da função anterior. A unica diferença é que o in permanece o mesmo e o out muda.
    Não é necessario no de cardinalidade
    '''
    uid +=1
    wire = ET.Element("Wire", UId = str(uid) )
    ET.SubElement(wire, "NameCon", UId=str(id_antes),Name="out")
    for x in range(len(lista_depois)):
        id_depois = lista_depois[x][-1][-1]
        ET.SubElement(wire, "NameCon", UId=str(id_depois), Name="in")
    ET.indent(wire, space = " ")
    xml_string = ET.tostring(wire, encoding="unicode") +"\n"
    lista_wires.append(xml_string)

    return(uid)


'''
Definição de como essas funções de preeenchimento de fio serão realizadas e implementadas no xml do TIA
'''
def contar_frequencia_nos_com_card(dados_ladder):
    contagem_entradas = {}
    contagem_saidas = {}

    for item in dados_ladder:
        # Se o primeiro elemento for 'card', pula para a próxima iteração do loop
        if item[0] == 'card':
            continue
        
        # Pega o número do nó de entrada (índice 1) e de saída (índice 2)
        no_in = item[1]
        no_out = item[2]
        
        # A chave é o nó, o valor é a quantidade de vezes (soma 1 a cada aparição)
        contagem_entradas[no_in] = contagem_entradas.get(no_in, 0) + 1
        contagem_saidas[no_out] = contagem_saidas.get(no_out, 0) + 1

    return contagem_entradas, contagem_saidas

def preencher_fios(uid,lista_com_cardinalidade):
    lista_aux = []
    lista_conferencia =[]
    identificou = [] #avalia se a função que identifica elemento e nome ja foi utilizada
    lista_l11 = []

    dic_entrada,dic_saida = contar_frequencia_nos_com_card(lista_com_cardinalidade)
    
    for x in range(len(lista_com_cardinalidade)):
        if lista_com_cardinalidade[x][0] =='card' or (dic_entrada[lista_com_cardinalidade[x][1]]== 0 and dic_saida[lista_com_cardinalidade[x][2]]== 1):
            continue
        else:
            if dic_entrada[lista_com_cardinalidade[x][1]]== 1 and dic_saida[lista_com_cardinalidade[x][2]]== 1:
                '''fazer o 1x1'''
                lista_l11.append(lista_com_cardinalidade[x])
                a = 0
                n = 1
                while a == 0 and (x + n) < len(lista_com_cardinalidade):
                    if lista_com_cardinalidade[x+n][0] == 'card':
                        n = n+1
                    else:
                        lista_l11.append(lista_com_cardinalidade[x+n])
                        a +=1
                uid = referenciar_nome_tipo(uid,lista_l11[0],lista_conferencia)
                #uid = fio_conect(uid, lista_l11[0][-1][-1], lista_l11[1][-1][-1])
                if len(lista_l11) > 1:
                    uid = fio_conect(uid,lista_l11[0][-1][-1],lista_l11[1][-1][-1])
                    #ta faltando confirmar se ja foi feito o identcon
                    uid = referenciar_nome_tipo(uid,lista_l11[1],lista_conferencia)
                lista_l11 = []   
            elif dic_entrada[lista_com_cardinalidade[x][1]] > 1 and dic_saida[lista_com_cardinalidade[x][2]]== 1:
                '''Fazer o multiplas entradas com cardinalidade'''
                if lista_com_cardinalidade[x-1][1] != lista_com_cardinalidade[x][1]:
                    contador_repeticoes_ordenado =1
                    inp_max = 1
                    parar = 0
                    while parar == 0:
                        if lista_com_cardinalidade[x+contador_repeticoes_ordenado][1] != lista_com_cardinalidade[x][1] or lista_com_cardinalidade[1] == 'card':
                            parar = 1
                        else:
                            inp_max += 1
                            contador_repeticoes_ordenado += 1
                    maximo_inputs = inp_max
                    if maximo_inputs == 1:
                        lista_l11 = []
                        lista_l11.append(lista_com_cardinalidade[x])
                        lista_l11.append(lista_com_cardinalidade[x+1])
                        uid = referenciar_nome_tipo(uid,lista_l11[0],lista_conferencia)
                        uid = fio_conect(uid,lista_l11[0][-1][-1],lista_l11[1][-1][-1])
                        #ta faltando confirmar se ja foi feito o identcon
                        uid = referenciar_nome_tipo(uid,lista_l11[1],lista_conferencia)
                        lista_l11 = []
                    else:
                        a = 0
                        n = 0
                        id_no_card = lista_com_cardinalidade[x+maximo_inputs][-1][0]
                        while a < maximo_inputs:
                            uid = referenciar_nome_tipo(uid,lista_com_cardinalidade[x+n],lista_conferencia)
                            uid = fio_conect_varios_esquerda(uid,lista_com_cardinalidade[x+n],id_no_card,n+1)
                            a = a+1
                            n = n+1
                        uid = fio_conect(uid,id_no_card,lista_com_cardinalidade[x+maximo_inputs+1][-1][-1])
                        uid = referenciar_nome_tipo(uid,lista_com_cardinalidade[x+maximo_inputs+1],lista_conferencia)


            elif dic_entrada[lista_com_cardinalidade[x][1]] == 1 and dic_saida[lista_com_cardinalidade[x][2]] > 1:
                '''Fazer o multiplas saidas'''
                if lista_com_cardinalidade[x-1][2] != lista_com_cardinalidade[x][2]:
                    lista_l11 = []
                    
                    # CORREÇÃO DE LÓGICA: Pegar a quantidade de saídas no nó (índice 2)
                    maximo_outputs = dic_saida[lista_com_cardinalidade[x][2]]
                    
                    a = 0
                    n = 1 # 'n' vai caminhar pela lista original
                    
                    # 1. Separar apenas os componentes reais, pulando os 'card'
                    while a < maximo_outputs and (x + n) < len(lista_com_cardinalidade):
                        if lista_com_cardinalidade[x+n][0] == 'card':
                            n += 1
                            continue # Pula o 'card' e volta pro início do while
                        
                        # Se for componente, guarda na lista_l11
                        lista_l11.append(lista_com_cardinalidade[x+n])
                        a += 1
                        n += 1
                    
                    # 2. Faz a conexão da esquerda com todos os elementos limpos à direita
                    uid = fio_conect_varios_direita(uid, lista_com_cardinalidade[x][-1][-1], lista_l11)
                    
                    # 3. Faz a referência de tipo só com os componentes válidos
                    for componente in lista_l11:
                        uid = referenciar_nome_tipo(uid, componente, lista_conferencia)

                    lista_l11 = []

    return(lista_wires)

            

def fios_master(uid,lista_com_cardinalidade,lista_geral_de_fios):
    lista_wires.clear()
    uid = barra_energizada(uid,lista_com_cardinalidade)
    lista_geral_de_fios.append([])
    inicio_wires = ["<Wires>"]
    fim_wires = ["</Wires>"]
    meio_wires= preencher_fios(uid,lista_com_cardinalidade)
    lista_geral_de_fios = inicio_wires  + meio_wires + fim_wires
    return(lista_geral_de_fios)






In [92]:
b = fios_master(a[-1],a[0],[])
b

['<Wires>',
 '</Wires>',
 '<Wire UId="48">\n  <IdentCon UId="21" />\n  <NameCon UId="33" Name="operand" />\n</Wire>',
 '<Wire UId="49">\n  <NameCon UId="33" Name="out" />\n  <NameCon UId="34" Name="in" />\n</Wire>',
 '<Wire UId="50">\n  <IdentCon UId="22" />\n  <NameCon UId="34" Name="operand" />\n</Wire>',
 '<Wire UId="51"><NameCon UId="34" Name="out" /><NameCon UId="35" Name="in" /><NameCon UId="36" Name="in" /><NameCon UId="38" Name="in" /></Wire>',
 '<Wire UId="52">\n  <IdentCon UId="23" />\n  <NameCon UId="35" Name="operand" />\n</Wire>',
 '<Wire UId="53">\n  <IdentCon UId="24" />\n  <NameCon UId="36" Name="operand" />\n</Wire>',
 '<Wire UId="54">\n  <IdentCon UId="25" />\n  <NameCon UId="38" Name="operand" />\n</Wire>',
 '<Wire UId="55">\n  <IdentCon UId="26" />\n  <NameCon UId="39" Name="operand" />\n</Wire>',
 '<Wire UId="56">\n  <NameCon UId="39" Name="out" />\n  <NameCon UId="40" Name="in" />\n</Wire>',
 '<Wire UId="57">\n  <IdentCon UId="27" />\n  <NameCon UId="40" Name="ope

In [ ]:
''' elif dic_entrada[lista_com_cardinalidade[x][1]] == 1 and dic_saida[lista_com_cardinalidade[x][2]]> 1:
                Fazer o multiplas saidas
                if lista_com_cardinalidade[x-1][2] != lista_com_cardinalidade[x][2]:
                    lista_l11 = []
                    maximo_outputs = dic_entrada[lista_com_cardinalidade[x][1]]
                    a = 0
                    n = 0
                    while a < maximo_outputs:
                        lista_l11.append(lista_com_cardinalidade[x+a+1])
                        a = a+1
                    a = 0
                    uid = fio_conect_varios_direita(uid,lista_com_cardinalidade[x][-1][-1],lista_l11)
                    while a< maximo_outputs:
                        uid = referenciar_nome_tipo(uid,lista_com_cardinalidade[x+a+1],lista_conferencia)
                        a = a+1
                 lista_l11 = []'''

In [31]:
'''
Bloco network
Agregação dos xml de part, wire e cabeçalhos
'''
def network_simples(lista, nome_bloco):
    lista_final = []
    
    data_atual = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S.%fZ')
    cabecalho = f"""<?xml version="1.0" encoding="utf-8"?>
<Document>
  <Engineering version="V17" />
  <DocumentInfo>
    <Created>{data_atual}</Created>
    <ExportSetting>None</ExportSetting>
    <InstalledProducts>
      <Product>
        <DisplayName>Totally Integrated Automation Portal</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </Product>
      <OptionPackage>
        <DisplayName>TIA Portal Openness</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </OptionPackage>
      <OptionPackage>
        <DisplayName>TIA Portal Version Control Interface</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </OptionPackage>
      <Product>
        <DisplayName>STEP 7 Professional</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </Product>
      <OptionPackage>
        <DisplayName>STEP 7 Safety</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </OptionPackage>
      <Product>
        <DisplayName>WinCC Professional</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </Product>
    </InstalledProducts>
  </DocumentInfo>
  <SW.Blocks.FC ID="0">
    <AttributeList>
      <Interface><Sections xmlns="http://www.siemens.com/automation/Openness/SW/Interface/v5">
  <Section Name="Input" />
  <Section Name="Output" />
  <Section Name="InOut" />
  <Section Name="Temp">
    <Member Name="cont2" Datatype="Bool" />
  </Section>
  <Section Name="Constant" />
  <Section Name="Return">
    <Member Name="Ret_Val" Datatype="Void" />
  </Section>
</Sections></Interface>
      <MemoryLayout>Optimized</MemoryLayout>
      <Name>{nome_bloco}</Name>
      <Number>1</Number>
      <ProgrammingLanguage>LAD</ProgrammingLanguage>
      <SetENOAutomatically>false</SetENOAutomatically>
    </AttributeList>
    <ObjectList>
      <MultilingualText ID="1" CompositionName="Comment">
        <ObjectList>
          <MultilingualTextItem ID="2" CompositionName="Items">
            <AttributeList>
              <Culture>en-US</Culture>
              <Text />
            </AttributeList>
          </MultilingualTextItem>
        </ObjectList>
      </MultilingualText>
      <SW.Blocks.CompileUnit ID="3" CompositionName="CompileUnits">
        <AttributeList>
          <NetworkSource><FlgNet xmlns="http://www.siemens.com/automation/Openness/SW/NetworkSource/FlgNet/v4">"""
    rodape = """</FlgNet></NetworkSource>
          <ProgrammingLanguage>LAD</ProgrammingLanguage>
        </AttributeList>
        <ObjectList>
          <MultilingualText ID="4" CompositionName="Comment">
            <ObjectList>
              <MultilingualTextItem ID="5" CompositionName="Items">
                <AttributeList>
                  <Culture>en-US</Culture>
                  <Text />
                </AttributeList>
              </MultilingualTextItem>
            </ObjectList>
          </MultilingualText>
          <MultilingualText ID="6" CompositionName="Title">
            <ObjectList>
              <MultilingualTextItem ID="7" CompositionName="Items">
                <AttributeList>
                  <Culture>en-US</Culture>
                  <Text />
                </AttributeList>
              </MultilingualTextItem>
            </ObjectList>
          </MultilingualText>
        </ObjectList>
      </SW.Blocks.CompileUnit>
      <MultilingualText ID="8" CompositionName="Title">
        <ObjectList>
          <MultilingualTextItem ID="9" CompositionName="Items">
            <AttributeList>
              <Culture>en-US</Culture>
              <Text />
            </AttributeList>
          </MultilingualTextItem>
        </ObjectList>
      </MultilingualText>
    </ObjectList>
  </SW.Blocks.FC>
</Document>"""
    with open(str(nome_bloco)+'.xml','w',encoding="utf-8") as arquivo:
      arquivo.write(cabecalho + "\n")
      part_network_simples = parts_final(lista[0],[])
      #for item in range(len(part_network_composto[1])):
      arquivo.writelines(part_network_simples[1][0])
      wire_network_simples=fios_master(part_network_simples[-1],part_network_simples[0],[])
      arquivo.writelines(wire_network_simples)
      arquivo.write(rodape)
    return('Finalizado')

def converter_para_hex(numero):
    """
    Converte qualquer número inteiro (ou texto contendo um número) 
    para o formato hexadecimal maiúsculo (ex: 31 vira '1F').
    """  
    return f"{numero:X}"

def gerar_cabecalho(nome_bloco, contador_id_inicial=0):
    # 1. Gerando a data e hora atual no formato do TIA Portal (UTC + Z)
    data_atual = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S.%fZ')
    
    # 2. Gerando os IDs em hexadecimal para o cabeçalho
    # O TIA Portal usa 4 IDs sequenciais só na abertura do arquivo
    id_fc      = converter_para_hex(contador_id_inicial)       # Ex: "0"
    id_comment = converter_para_hex(contador_id_inicial + 1)   # Ex: "1"
    id_item    = converter_para_hex(contador_id_inicial + 2)   # Ex: "2"
    id_compile = converter_para_hex(contador_id_inicial + 3)   # Ex: "3"
    
    # 3. Montando o cabeçalho injetando as variáveis (F-string)
    cabecalho = f"""<?xml version="1.0" encoding="utf-8"?>
<Document>
  <Engineering version="V17" />
  <DocumentInfo>
    <Created>{data_atual}</Created>
    <ExportSetting>None</ExportSetting>
    <InstalledProducts>
      <Product>
        <DisplayName>Totally Integrated Automation Portal</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </Product>
      <OptionPackage>
        <DisplayName>TIA Portal Openness</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </OptionPackage>
      <OptionPackage>
        <DisplayName>TIA Portal Version Control Interface</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </OptionPackage>
      <Product>
        <DisplayName>STEP 7 Professional</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </Product>
      <OptionPackage>
        <DisplayName>STEP 7 Safety</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </OptionPackage>
      <Product>
        <DisplayName>WinCC Professional</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </Product>
    </InstalledProducts>
  </DocumentInfo>
  <SW.Blocks.FC ID="{id_fc}">
    <AttributeList>
      <Interface><Sections xmlns="http://www.siemens.com/automation/Openness/SW/Interface/v5">
  <Section Name="Input" />
  <Section Name="Output" />
  <Section Name="InOut" />
  <Section Name="Temp">
    <Member Name="cont2" Datatype="Bool" />
  </Section>
  <Section Name="Constant" />
  <Section Name="Return">
    <Member Name="Ret_Val" Datatype="Void" />
  </Section>
</Sections></Interface>
      <MemoryLayout>Optimized</MemoryLayout>
      <Name>{nome_bloco}</Name>
      <Number>1</Number>
      <ProgrammingLanguage>LAD</ProgrammingLanguage>
      <SetENOAutomatically>false</SetENOAutomatically>
    </AttributeList>
    <ObjectList>
      <MultilingualText ID="{id_comment}" CompositionName="Comment">
        <ObjectList>
          <MultilingualTextItem ID="{id_item}" CompositionName="Items">
            <AttributeList>
              <Culture>en-US</Culture>
              <Text />
            </AttributeList>
          </MultilingualTextItem>
        </ObjectList>
      </MultilingualText>
      <SW.Blocks.CompileUnit ID="{id_compile}" CompositionName="CompileUnits">
        <AttributeList>
          <NetworkSource><FlgNet xmlns="http://www.siemens.com/automation/Openness/SW/NetworkSource/FlgNet/v4">"""

    # Retorna o texto do cabeçalho E o próximo número disponível para não perdermos a contagem
    proximo_id_disponivel = contador_id_inicial + 4 
    
    return cabecalho, proximo_id_disponivel

def gerar_separador_network(contador_id_atual):
    """
    Gera o bloco XML que fecha um network e abre o próximo.
    Recebe o ID atual, consome 5 IDs para as tags e retorna o texto e o próximo ID.
    """
    # Calculando os próximos 5 IDs em hexadecimal
    id_comment = converter_para_hex(contador_id_atual)       # Ex: "4"
    id_item_c  = converter_para_hex(contador_id_atual + 1)   # Ex: "5"
    id_title   = converter_para_hex(contador_id_atual + 2)   # Ex: "6"
    id_item_t  = converter_para_hex(contador_id_atual + 3)   # Ex: "7"
    id_compile = converter_para_hex(contador_id_atual + 4)   # Ex: "8" (Abre o próximo network)
    
    # Montando a string com F-string
    final_network = f"""</FlgNet></NetworkSource>
          <ProgrammingLanguage>LAD</ProgrammingLanguage>
        </AttributeList>
        <ObjectList>
          <MultilingualText ID="{id_comment}" CompositionName="Comment">
            <ObjectList>
              <MultilingualTextItem ID="{id_item_c}" CompositionName="Items">
                <AttributeList>
                  <Culture>en-US</Culture>
                  <Text />
                </AttributeList>
              </MultilingualTextItem>
            </ObjectList>
          </MultilingualText>
          <MultilingualText ID="{id_title}" CompositionName="Title">
            <ObjectList>
              <MultilingualTextItem ID="{id_item_t}" CompositionName="Items">
                <AttributeList>
                  <Culture>en-US</Culture>
                  <Text />
                </AttributeList>
              </MultilingualTextItem>
            </ObjectList>
          </MultilingualText>
        </ObjectList>
      </SW.Blocks.CompileUnit>
      <SW.Blocks.CompileUnit ID="{id_compile}" CompositionName="CompileUnits">
        <AttributeList>
          <NetworkSource><FlgNet xmlns="http://www.siemens.com/automation/Openness/SW/NetworkSource/FlgNet/v4">"""

    # O próximo ID disponível será o contador atual + 5
    proximo_id_disponivel = contador_id_atual + 5
    
    return final_network, proximo_id_disponivel


def gerar_rodape(contador_id_atual):
    """
    Gera o bloco XML final que encerra o último network e o documento.
    Recebe o ID atual, consome 6 IDs e retorna a string final.
    """
    # Calculando os últimos 6 IDs em hexadecimal
    id_comment  = converter_para_hex(contador_id_atual)      # Ex: "1D"
    id_item_c   = converter_para_hex(contador_id_atual + 1)  # Ex: "1E"
    id_title    = converter_para_hex(contador_id_atual + 2)  # Ex: "1F"
    id_item_t   = converter_para_hex(contador_id_atual + 3)  # Ex: "20"
    
    # IDs finais do bloco da função (FC)
    id_fc_title = converter_para_hex(contador_id_atual + 4)  # Ex: "21"
    id_fc_item  = converter_para_hex(contador_id_atual + 5)  # Ex: "22"
    
    # Montando a string final com F-string
    rodape = f"""</FlgNet></NetworkSource>
          <ProgrammingLanguage>LAD</ProgrammingLanguage>
        </AttributeList>
        <ObjectList>
          <MultilingualText ID="{id_comment}" CompositionName="Comment">
            <ObjectList>
              <MultilingualTextItem ID="{id_item_c}" CompositionName="Items">
                <AttributeList>
                  <Culture>en-US</Culture>
                  <Text />
                </AttributeList>
              </MultilingualTextItem>
            </ObjectList>
          </MultilingualText>
          <MultilingualText ID="{id_title}" CompositionName="Title">
            <ObjectList>
              <MultilingualTextItem ID="{id_item_t}" CompositionName="Items">
                <AttributeList>
                  <Culture>en-US</Culture>
                  <Text />
                </AttributeList>
              </MultilingualTextItem>
            </ObjectList>
          </MultilingualText>
        </ObjectList>
      </SW.Blocks.CompileUnit>
      <MultilingualText ID="{id_fc_title}" CompositionName="Title">
        <ObjectList>
          <MultilingualTextItem ID="{id_fc_item}" CompositionName="Items">
            <AttributeList>
              <Culture>en-US</Culture>
              <Text />
            </AttributeList>
          </MultilingualTextItem>
        </ObjectList>
      </MultilingualText>
    </ObjectList>
  </SW.Blocks.FC>
</Document>"""

    return rodape

def network_composto(lista,nome_bloco='Block_1'):
  with open(str(nome_bloco)+'.xml','w',encoding="utf-8") as arquivo:
    cabecalho = gerar_cabecalho(nome_bloco, contador_id_inicial=0)
    arquivo.write(cabecalho[0] + "\n")
    for x in range(len(lista)-1):
      part_network_composto = parts_final(lista[x],[])
      #for item in range(len(part_network_composto[1])):
      arquivo.writelines(part_network_composto[1][0])
      wire_network_composto=fios_master(part_network_composto[-1],part_network_composto[0],[])
      arquivo.writelines(wire_network_composto)
      if x == 0:
        meio = gerar_separador_network(cabecalho[1])
      else:
         meio = gerar_separador_network(meio[1])
      arquivo.write(meio[0])
    part_network_composto = parts_final(lista[-1],[])
    arquivo.writelines(part_network_composto[1][0])
    wire_network_composto=fios_master(part_network_composto[-1],part_network_composto[0],[])
    arquivo.writelines(wire_network_composto)
    fim = gerar_rodape(meio[1])
    arquivo.write(fim)
  return('Finalizado')
         
         


        




      

In [46]:
def obter_cabecalho_tags():
    return """<?xml version="1.0" encoding="utf-8"?>
<Document>
  <Engineering version="V17" />
  <DocumentInfo>
    <Created>2026-05-12T13:54:25.9248339Z</Created>
    <ExportSetting>None</ExportSetting>
    <InstalledProducts>
      <Product>
        <DisplayName>Totally Integrated Automation Portal</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </Product>
      <OptionPackage>
        <DisplayName>TIA Portal Openness</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </OptionPackage>
      <OptionPackage>
        <DisplayName>TIA Portal Version Control Interface</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </OptionPackage>
      <Product>
        <DisplayName>STEP 7 Professional</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </Product>
      <OptionPackage>
        <DisplayName>STEP 7 Safety</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </OptionPackage>
      <Product>
        <DisplayName>WinCC Professional</DisplayName>
        <DisplayVersion>V17</DisplayVersion>
      </Product>
    </InstalledProducts>
  </DocumentInfo>
  <SW.Tags.PlcTagTable ID="0">
    <AttributeList>
      <Name>Default tag table python</Name>
    </AttributeList>
    <ObjectList>"""

def obter_rodape_tags():
    return """  

    </ObjectList>
  </SW.Tags.PlcTagTable>
</Document>"""

def gerar_xml_tags_automático(lista, lista_tipos):
    # Lista auxiliar para controle de duplicidade nesta execução
    nomes_adicionados_agora = []
    
    xml_corpo = ""
    
    # Contadores para endereço %M
    byte = 0
    bit = 0
    
    id_atual = 1 

    if len(lista) != 1:
        lista_aux_tag = []
        for y in range(len(lista)):
            lista_aux_tag += lista[y]
        lista = lista_aux_tag
    
    for x in range(len(lista)):
        # Regra: Só adiciona se NÃO estiver na global E NÃO estiver na local
        if  lista[x][0][0] not in nomes_adicionados_agora:
            
            nomes_adicionados_agora.append(lista[x][0][0])
            
        
            hex_id_tag = hex(id_atual)[2:].upper()
            hex_id_comm = hex(id_atual + 1)[2:].upper()
            hex_id_item = hex(id_atual + 2)[2:].upper()
            
            endereco = f"%M{byte}.{bit}"
            
    
            tag_template = f"""
      <SW.Tags.PlcTag ID="{hex_id_tag}" CompositionName="Tags">
        <AttributeList>
          <DataTypeName>Bool</DataTypeName>
          <LogicalAddress>{endereco}</LogicalAddress>
          <Name>{lista[x][0][0]}</Name>
        </AttributeList>
        <ObjectList>
          <MultilingualText ID="{hex_id_comm}" CompositionName="Comment">
            <ObjectList>
              <MultilingualTextItem ID="{hex_id_item}" CompositionName="Items">
                <AttributeList>
                  <Culture>en-US</Culture>
                  <Text />
                </AttributeList>
              </MultilingualTextItem>
            </ObjectList>
          </MultilingualText>
        </ObjectList>
      </SW.Tags.PlcTag>"""
            
            xml_corpo += tag_template
            
            # Incrementos
            id_atual += 3 # Salto de 3 IDs conforme seu XML
            bit += 1
            if bit > 7:
                bit = 0
                byte += 1
    xml_final=  obter_cabecalho_tags() + xml_corpo + "\n" + obter_rodape_tags()   
    with open("Default tag table.xml", "w", encoding="utf-8") as f:
        f.write(xml_final)  
 


    

In [33]:
def criar_network(lista,nome_bloco='Block_1'):
    if len(lista) == 1:
       network_simples(lista,nome_bloco)
       gerar_xml_tags_automático(lista, [])
       
    else:
       network_composto(lista,nome_bloco)
       gerar_xml_tags_automático(lista, [])
  

In [18]:
#len(dados_ladder_otimizados)
criar_network(dados_ladder_otimizados,'bloquinho')

--------DEBUG--------
['<Wire UId="47">\n  <Powerrail />\n  <NameCon UId="33" Name="in" />\n  <NameCon UId="35" Name="in" />\n  <NameCon UId="36" Name="in" />\n  <NameCon UId="39" Name="in" />\n</Wire>\n']


In [23]:
net1 = [[('cont1','contato','1',0),0,1],
        [('bobina1','bobina','1',0),1,2]]
net2 = [[('cont8', 'contato', '1', 0), 0, 1],[('bobina2', 'bobina', '1', 0), 1, 2]]
net3 = []
net3.append(net1)
net3.append(net2)
net3

[[[('cont1', 'contato', '1', 0), 0, 1], [('bobina1', 'bobina', '1', 0), 1, 2]],
 [[('cont8', 'contato', '1', 0), 0, 1], [('bobina2', 'bobina', '1', 0), 1, 2]]]

In [30]:
net1 +net2

[[('cont1', 'contato', '1', 0), 0, 1],
 [('bobina1', 'bobina', '1', 0), 1, 2],
 [('cont8', 'contato', '1', 0), 0, 1],
 [('bobina2', 'bobina', '1', 0), 1, 2]]

In [47]:
criar_network(net3,'exemplo')

--------DEBUG--------
['<Wire UId="25">\n  <Powerrail />\n  <NameCon UId="23" Name="in" />\n</Wire>\n']
--------DEBUG--------
['<Wire UId="25">\n  <Powerrail />\n  <NameCon UId="23" Name="in" />\n</Wire>\n']


In [26]:
pre_xml[2]

47

In [2]:
a =[11]
b = [22]
c = [44]
d = a+b+c
d

[11, 22, 44]

In [27]:
preencher_fios(pre_xml[2],pre_xml[0])

'Network finalizada'

In [29]:
print(lista_wires)

['<Wire UId="48">\n  <IdentCon UId="21" />\n  <NameCon UId="33" Name="operand" />\n</Wire>', '<Wire UId="49">\n  <NameCon UId="33" Name="out" />\n  <NameCon UId="34" Name="in" />\n</Wire>', '<Wire UId="50">\n  <IdentCon UId="22" />\n  <NameCon UId="34" Name="operand" />\n</Wire>', '<Wire UId="51"><NameCon UId="34" Name="out" /><NameCon UId="35" Name="in" /><NameCon UId="36" Name="in" /><NameCon UId="38" Name="in" /></Wire>', '<Wire UId="52">\n  <IdentCon UId="23" />\n  <NameCon UId="35" Name="operand" />\n</Wire>', '<Wire UId="53">\n  <IdentCon UId="24" />\n  <NameCon UId="36" Name="operand" />\n</Wire>', '<Wire UId="54">\n  <IdentCon UId="25" />\n  <NameCon UId="38" Name="operand" />\n</Wire>', '<Wire UId="55">\n  <IdentCon UId="26" />\n  <NameCon UId="39" Name="operand" />\n</Wire>', '<Wire UId="56">\n  <NameCon UId="39" Name="out" />\n  <NameCon UId="40" Name="in" />\n</Wire>', '<Wire UId="57">\n  <IdentCon UId="27" />\n  <NameCon UId="40" Name="operand" />\n</Wire>', '<Wire UId="58

In [30]:
for wire in lista_wires:
    print(wire)

<Wire UId="48">
  <IdentCon UId="21" />
  <NameCon UId="33" Name="operand" />
</Wire>
<Wire UId="49">
  <NameCon UId="33" Name="out" />
  <NameCon UId="34" Name="in" />
</Wire>
<Wire UId="50">
  <IdentCon UId="22" />
  <NameCon UId="34" Name="operand" />
</Wire>
<Wire UId="51"><NameCon UId="34" Name="out" /><NameCon UId="35" Name="in" /><NameCon UId="36" Name="in" /><NameCon UId="38" Name="in" /></Wire>
<Wire UId="52">
  <IdentCon UId="23" />
  <NameCon UId="35" Name="operand" />
</Wire>
<Wire UId="53">
  <IdentCon UId="24" />
  <NameCon UId="36" Name="operand" />
</Wire>
<Wire UId="54">
  <IdentCon UId="25" />
  <NameCon UId="38" Name="operand" />
</Wire>
<Wire UId="55">
  <IdentCon UId="26" />
  <NameCon UId="39" Name="operand" />
</Wire>
<Wire UId="56">
  <NameCon UId="39" Name="out" />
  <NameCon UId="40" Name="in" />
</Wire>
<Wire UId="57">
  <IdentCon UId="27" />
  <NameCon UId="40" Name="operand" />
</Wire>
<Wire UId="58"><NameCon UId="40" Name="out" /><NameCon UId="42" Name="in

In [35]:
def gerador_de_id(inicio=0):
    """Gera uma sequência infinita de IDs em hexadecimal maiúsculo."""
    contador = inicio
    while True:
        # Formata o número atual para Hexadecimal Maiúsculo (:X)
        yield f"{contador:X}"
        contador += 1

# --- Como usar ---
# 1. Inicializamos o gerador
novo_id = gerador_de_id()

# 2. Usamos a função next() toda vez que precisarmos de um novo ID
print(next(novo_id))  # Saída: 0
print(next(novo_id))  # Saída: 1
print(next(novo_id))  # Saída: 2

# Se chamarmos várias vezes, ele vai chegar nas letras
for _ in range(100):
    print(next(novo_id)) 
# Vai imprimir: 3, 4, 5, 6, 7, 8, 9, A, B, C

0
1
2
3
4
5
6
7
8
9
A
B
C
D
E
F
10
11
12
13
14
15
16
17
18
19
1A
1B
1C
1D
1E
1F
20
21
22
23
24
25
26
27
28
29
2A
2B
2C
2D
2E
2F
30
31
32
33
34
35
36
37
38
39
3A
3B
3C
3D
3E
3F
40
41
42
43
44
45
46
47
48
49
4A
4B
4C
4D
4E
4F
50
51
52
53
54
55
56
57
58
59
5A
5B
5C
5D
5E
5F
60
61
62
63
64
65
66


In [21]:
pre_xml[0]

[[('cont1', 'contato', '1', 0), 0, 1, [21, 33]],
 [('cont9', 'contato', '1', 1), 1, 2, [22, 34]],
 [('cont3', 'contato', '1', 0), 0, 2, [23, 35]],
 [('cont4', 'contato', '1', 0), 0, 2, [24, 36]],
 ['card', 2, 3, [37]],
 [('cont5', 'contato', '1', 0), 2, 3, [25, 38]],
 [('cont9', 'contato', '1', 0), 0, 4, [26, 39]],
 [('cont10', 'contato', '1', 0), 4, 3, [27, 40]],
 ['card', 3, 2, [41]],
 [('cont6', 'contato', '1', 0), 3, 5, [28, 42]],
 [('bobina1', 'contato', '1', 0), 5, 6, [29, 43]],
 [('cont7', 'contato', '1', 0), 3, 7, [30, 44]],
 [('cont8', 'contato', '1', 0), 7, 8, [31, 45]],
 [('bobina2', 'contato', '1', 0), 8, 9, [32, 46]]]

In [25]:
a = barra_energizada(47,pre_xml[0])

[[], '<Wire UId="47">\n  <Powerrail />\n  <NameCon UId="33" Name="in" />\n  <NameCon UId="35" Name="in" />\n  <NameCon UId="36" Name="in" />\n  <NameCon UId="39" Name="in" />\n</Wire>']


In [27]:
print(a[0][1])

<Wire UId="47">
  <Powerrail />
  <NameCon UId="33" Name="in" />
  <NameCon UId="35" Name="in" />
  <NameCon UId="36" Name="in" />
  <NameCon UId="39" Name="in" />
</Wire>
